# AutoML Framework API Demo

This notebook demonstrates how to:
1. Login as admin to the AutoML Framework API
2. Create a time series dataset
3. Upload the dataset
4. Create a time series forecasting project/experiment
5. Monitor the experiment progress
6. Retrieve results

## Prerequisites
- AutoML Framework API running on http://localhost:8000
- Admin credentials: username='admin', password='secret'

In [ ]:
# Import required libraries
import requests
import pandas as pd
import numpy as np
import json
import time
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns
from io import StringIO
import warnings
warnings.filterwarnings('ignore')

# Set up plotting
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("Libraries imported successfully!")

## 1. API Configuration and Authentication

In [ ]:
# API Configuration
API_BASE_URL = "http://localhost:8000"
ADMIN_USERNAME = "admin"
ADMIN_PASSWORD = "secret"

# Global session for API calls
session = requests.Session()

def login(username: str, password: str) -> str:
    """Login to the API and return access token."""
    login_url = f"{API_BASE_URL}/api/v1/auth/login"
    
    # Prepare form data for OAuth2PasswordRequestForm
    login_data = {
        "username": username,
        "password": password
    }
    
    headers = {
        "Content-Type": "application/x-www-form-urlencoded"
    }
    
    response = session.post(login_url, data=login_data, headers=headers)
    
    if response.status_code == 200:
        token_data = response.json()
        access_token = token_data["access_token"]
        
        # Set authorization header for future requests
        session.headers.update({"Authorization": f"Bearer {access_token}"})
        
        print(f"✅ Successfully logged in as {username}")
        print(f"Token type: {token_data['token_type']}")
        return access_token
    else:
        print(f"❌ Login failed: {response.status_code}")
        print(f"Error: {response.text}")
        return None

# Login as admin
access_token = login(ADMIN_USERNAME, ADMIN_PASSWORD)

if access_token:
    print(f"\n🔑 Access token (first 50 chars): {access_token[:50]}...")
else:
    raise Exception("Failed to authenticate. Please check your credentials and API status.")

## 2. Check API Health and Configuration

In [ ]:
# Check API health
health_response = session.get(f"{API_BASE_URL}/health")
if health_response.status_code == 200:
    health_data = health_response.json()
    print("🏥 API Health Status:")
    print(f"  Status: {health_data['status']}")
    print(f"  Environment: {health_data['environment']}")
    print(f"  GPU Available: {health_data['gpu_available']}")
    print(f"  Database Available: {health_data['database_available']}")
    print(f"  Auth Backend: {health_data['authentication']['backend']}")
    print(f"  Auth Test: {health_data['authentication']['test_result']}")
else:
    print(f"❌ Health check failed: {health_response.status_code}")

# Check API configuration
config_response = session.get(f"{API_BASE_URL}/api/v1/config")
if config_response.status_code == 200:
    config_data = config_response.json()
    print("\n⚙️ API Configuration:")
    print(f"  Version: {config_data['version']}")
    print(f"  Supported Data Types: {config_data['supported_data_types']}")
    print(f"  Supported Task Types: {config_data['supported_task_types']}")
    print(f"  Max File Size: {config_data['max_file_size_mb']}MB")
else:
    print(f"❌ Config check failed: {config_response.status_code}")

## 3. Generate Sample Time Series Data

In [ ]:
def generate_time_series_data(n_points=1000, start_date='2020-01-01'):
    """Generate synthetic time series data for demonstration."""
    
    # Create date range
    dates = pd.date_range(start=start_date, periods=n_points, freq='D')
    
    # Generate base trend
    trend = np.linspace(100, 200, n_points)
    
    # Add seasonal component (yearly cycle)
    seasonal = 20 * np.sin(2 * np.pi * np.arange(n_points) / 365.25)
    
    # Add weekly pattern
    weekly = 10 * np.sin(2 * np.pi * np.arange(n_points) / 7)
    
    # Add noise
    noise = np.random.normal(0, 5, n_points)
    
    # Combine components
    values = trend + seasonal + weekly + noise
    
    # Add some external features
    temperature = 20 + 15 * np.sin(2 * np.pi * np.arange(n_points) / 365.25) + np.random.normal(0, 2, n_points)
    day_of_week = np.arange(n_points) % 7
    month = dates.month
    
    # Create DataFrame
    df = pd.DataFrame({
        'date': dates,
        'value': values,
        'temperature': temperature,
        'day_of_week': day_of_week,
        'month': month,
        'is_weekend': (day_of_week >= 5).astype(int)
    })
    
    return df

# Generate the dataset
print("📊 Generating synthetic time series data...")
ts_data = generate_time_series_data(n_points=800)

print(f"✅ Generated dataset with {len(ts_data)} data points")
print(f"Date range: {ts_data['date'].min()} to {ts_data['date'].max()}")
print(f"\nDataset shape: {ts_data.shape}")
print(f"\nColumns: {list(ts_data.columns)}")
print(f"\nFirst few rows:")
print(ts_data.head())

In [ ]:
# Visualize the time series data
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Main time series
axes[0, 0].plot(ts_data['date'], ts_data['value'], linewidth=1, alpha=0.8)
axes[0, 0].set_title('Time Series Values', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Date')
axes[0, 0].set_ylabel('Value')
axes[0, 0].grid(True, alpha=0.3)

# Temperature feature
axes[0, 1].plot(ts_data['date'], ts_data['temperature'], color='orange', linewidth=1, alpha=0.8)
axes[0, 1].set_title('Temperature Feature', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Date')
axes[0, 1].set_ylabel('Temperature')
axes[0, 1].grid(True, alpha=0.3)

# Distribution of values
axes[1, 0].hist(ts_data['value'], bins=50, alpha=0.7, color='skyblue', edgecolor='black')
axes[1, 0].set_title('Distribution of Values', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Value')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].grid(True, alpha=0.3)

# Correlation heatmap
numeric_cols = ['value', 'temperature', 'day_of_week', 'month', 'is_weekend']
corr_matrix = ts_data[numeric_cols].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, ax=axes[1, 1], cbar_kws={'shrink': 0.8})
axes[1, 1].set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

# Print basic statistics
print("\n📈 Dataset Statistics:")
print(ts_data.describe())

## 4. Upload Dataset to AutoML Framework

In [ ]:
# Save dataset to CSV for upload
csv_filename = "time_series_demo_data.csv"
ts_data.to_csv(csv_filename, index=False)
print(f"💾 Saved dataset to {csv_filename}")

def upload_dataset(file_path: str, name: str, description: str):
    """Upload dataset to the AutoML Framework."""
    upload_url = f"{API_BASE_URL}/api/v1/datasets/upload"
    
    # Prepare files and data for upload
    with open(file_path, 'rb') as f:
        files = {'file': (file_path, f, 'text/csv')}
        data = {
            'name': name,
            'description': description
        }
        
        # Remove Content-Type header to let requests set it automatically for multipart
        headers = {k: v for k, v in session.headers.items() if k.lower() != 'content-type'}
        
        response = requests.post(upload_url, files=files, data=data, headers=headers)
    
    return response

# Upload the dataset
print("📤 Uploading dataset to AutoML Framework...")
upload_response = upload_dataset(
    file_path=csv_filename,
    name="Time Series Demo Dataset",
    description="Synthetic time series data with trend, seasonality, and external features for forecasting demo"
)

if upload_response.status_code == 200:
    upload_data = upload_response.json()
    dataset_id = upload_data['dataset_id']
    print(f"✅ Dataset uploaded successfully!")
    print(f"Dataset ID: {dataset_id}")
    print(f"Filename: {upload_data['filename']}")
    print(f"Size: {upload_data['size_bytes']} bytes")
    print(f"Metadata: {json.dumps(upload_data['metadata'], indent=2)}")
else:
    print(f"❌ Dataset upload failed: {upload_response.status_code}")
    print(f"Error: {upload_response.text}")
    dataset_id = None

## 5. Create Time Series Forecasting Experiment

In [ ]:
def create_experiment(name: str, dataset_path: str, task_type: str, data_type: str, 
                     target_column: str, config: dict):
    """Create a new experiment."""
    create_url = f"{API_BASE_URL}/api/v1/experiments"
    
    experiment_data = {
        "name": name,
        "dataset_path": dataset_path,
        "task_type": task_type,
        "data_type": data_type,
        "target_column": target_column,
        "config": config
    }
    
    headers = {"Content-Type": "application/json"}
    headers.update({k: v for k, v in session.headers.items() if k.lower() != 'content-type'})
    
    response = requests.post(create_url, json=experiment_data, headers=headers)
    return response

if dataset_id:
    # Configure the time series forecasting experiment
    experiment_config = {
        "forecast_horizon": 30,  # Predict 30 days ahead
        "lookback_window": 60,   # Use 60 days of history
        "validation_split": 0.2,
        "max_trials": 10,        # Limit trials for demo
        "max_epochs": 50,        # Limit epochs for demo
        "features": {
            "date_column": "date",
            "value_column": "value",
            "external_features": ["temperature", "day_of_week", "month", "is_weekend"]
        },
        "model_types": ["lstm", "gru", "transformer"],
        "optimization_metric": "mae",  # Mean Absolute Error
        "early_stopping": {
            "patience": 10,
            "min_delta": 0.001
        }
    }
    
    print("🚀 Creating time series forecasting experiment...")
    experiment_response = create_experiment(
        name="Time Series Forecasting Demo",
        dataset_path=f"datasets/{dataset_id}/{csv_filename}",
        task_type="time_series_forecasting",
        data_type="time_series",
        target_column="value",
        config=experiment_config
    )
    
    if experiment_response.status_code == 200:
        experiment_data = experiment_response.json()
        experiment_id = experiment_data['id']
        print(f"✅ Experiment created successfully!")
        print(f"Experiment ID: {experiment_id}")
        print(f"Name: {experiment_data['name']}")
        print(f"Status: {experiment_data['status']}")
        print(f"Created at: {experiment_data['created_at']}")
        print(f"\nExperiment Configuration:")
        print(json.dumps(experiment_config, indent=2))
    else:
        print(f"❌ Experiment creation failed: {experiment_response.status_code}")
        print(f"Error: {experiment_response.text}")
        experiment_id = None
else:
    print("⚠️ Skipping experiment creation due to dataset upload failure")
    experiment_id = None

## 6. Monitor Experiment Progress

In [ ]:
def get_experiment_status(experiment_id: str):
    """Get current experiment status."""
    status_url = f"{API_BASE_URL}/api/v1/experiments/{experiment_id}"
    response = session.get(status_url)
    return response

def monitor_experiment(experiment_id: str, max_wait_time: int = 300, check_interval: int = 10):
    """Monitor experiment progress until completion or timeout."""
    start_time = time.time()
    
    print(f"🔍 Monitoring experiment {experiment_id}...")
    print(f"Max wait time: {max_wait_time} seconds")
    print(f"Check interval: {check_interval} seconds")
    print("\n" + "="*60)
    
    while time.time() - start_time < max_wait_time:
        response = get_experiment_status(experiment_id)
        
        if response.status_code == 200:
            data = response.json()
            status = data['status']
            progress = data.get('progress', {})
            
            elapsed_time = int(time.time() - start_time)
            timestamp = datetime.now().strftime("%H:%M:%S")
            
            print(f"[{timestamp}] Status: {status.upper()} | Elapsed: {elapsed_time}s")
            
            if progress:
                for key, value in progress.items():
                    if isinstance(value, float):
                        print(f"  {key}: {value:.2%}")
                    else:
                        print(f"  {key}: {value}")
            
            if status in ['completed', 'failed', 'cancelled']:
                print(f"\n🏁 Experiment {status.upper()}!")
                return data
            
            print("-" * 40)
            time.sleep(check_interval)
        else:
            print(f"❌ Failed to get experiment status: {response.status_code}")
            break
    
    print(f"\n⏰ Monitoring timeout reached ({max_wait_time}s)")
    return None

if experiment_id:
    # Monitor the experiment (with shorter timeout for demo)
    final_status = monitor_experiment(experiment_id, max_wait_time=180, check_interval=15)
    
    if final_status:
        print(f"\n📊 Final Experiment Status:")
        print(f"ID: {final_status['id']}")
        print(f"Name: {final_status['name']}")
        print(f"Status: {final_status['status']}")
        print(f"Created: {final_status['created_at']}")
        if final_status.get('completed_at'):
            print(f"Completed: {final_status['completed_at']}")
        if final_status.get('results'):
            print(f"\n🎯 Results Available: Yes")
        else:
            print(f"\n🎯 Results Available: No (experiment may still be running)")
else:
    print("⚠️ Skipping experiment monitoring due to creation failure")

## 7. Retrieve and Display Results

In [ ]:
def get_experiment_results(experiment_id: str):
    """Get detailed experiment results."""
    results_url = f"{API_BASE_URL}/api/v1/experiments/{experiment_id}/results"
    response = session.get(results_url)
    return response

if experiment_id:
    print(f"📈 Retrieving experiment results...")
    results_response = get_experiment_results(experiment_id)
    
    if results_response.status_code == 200:
        results_data = results_response.json()
        print(f"✅ Results retrieved successfully!")
        
        # Display results summary
        if 'best_model' in results_data:
            best_model = results_data['best_model']
            print(f"\n🏆 Best Model:")
            print(f"  Architecture: {best_model.get('architecture', 'N/A')}")
            print(f"  Score: {best_model.get('score', 'N/A')}")
            print(f"  Metric: {best_model.get('metric', 'N/A')}")
        
        if 'metrics' in results_data:
            metrics = results_data['metrics']
            print(f"\n📊 Performance Metrics:")
            for metric_name, metric_value in metrics.items():
                if isinstance(metric_value, float):
                    print(f"  {metric_name}: {metric_value:.4f}")
                else:
                    print(f"  {metric_name}: {metric_value}")
        
        if 'predictions' in results_data:
            predictions = results_data['predictions']
            print(f"\n🔮 Predictions: {len(predictions)} data points")
            
            # Convert predictions to DataFrame for visualization
            pred_df = pd.DataFrame(predictions)
            
            # Plot predictions vs actual values
            if 'actual' in pred_df.columns and 'predicted' in pred_df.columns:
                plt.figure(figsize=(12, 6))
                
                plt.subplot(1, 2, 1)
                plt.plot(pred_df['actual'], label='Actual', alpha=0.8)
                plt.plot(pred_df['predicted'], label='Predicted', alpha=0.8)
                plt.title('Time Series Predictions vs Actual')
                plt.xlabel('Time Step')
                plt.ylabel('Value')
                plt.legend()
                plt.grid(True, alpha=0.3)
                
                plt.subplot(1, 2, 2)
                plt.scatter(pred_df['actual'], pred_df['predicted'], alpha=0.6)
                plt.plot([pred_df['actual'].min(), pred_df['actual'].max()], 
                        [pred_df['actual'].min(), pred_df['actual'].max()], 
                        'r--', alpha=0.8)
                plt.title('Predicted vs Actual Values')
                plt.xlabel('Actual Values')
                plt.ylabel('Predicted Values')
                plt.grid(True, alpha=0.3)
                
                plt.tight_layout()
                plt.show()
        
        # Display full results (truncated for readability)
        print(f"\n📋 Full Results Summary:")
        print(json.dumps(results_data, indent=2, default=str)[:2000] + "..." if len(str(results_data)) > 2000 else json.dumps(results_data, indent=2, default=str))
        
    elif results_response.status_code == 404:
        print(f"⚠️ Results not yet available. Experiment may still be running.")
    else:
        print(f"❌ Failed to retrieve results: {results_response.status_code}")
        print(f"Error: {results_response.text}")
else:
    print("⚠️ Skipping results retrieval due to experiment creation failure")

## 8. List All Experiments

In [ ]:
def list_experiments():
    """List all experiments."""
    list_url = f"{API_BASE_URL}/api/v1/experiments"
    response = session.get(list_url)
    return response

print("📋 Listing all experiments...")
list_response = list_experiments()

if list_response.status_code == 200:
    experiments_data = list_response.json()
    experiments = experiments_data.get('experiments', [])
    total = experiments_data.get('total', 0)
    
    print(f"✅ Found {total} experiment(s)")
    
    if experiments:
        print(f"\n📊 Experiments Summary:")
        print("-" * 80)
        print(f"{'ID':<20} {'Name':<30} {'Status':<12} {'Created':<20}")
        print("-" * 80)
        
        for exp in experiments:
            exp_id = exp['id'][:18] + '...' if len(exp['id']) > 20 else exp['id']
            exp_name = exp['name'][:28] + '...' if len(exp['name']) > 30 else exp['name']
            exp_status = exp['status']
            exp_created = exp['created_at'][:19] if exp['created_at'] else 'N/A'
            
            print(f"{exp_id:<20} {exp_name:<30} {exp_status:<12} {exp_created:<20}")
        
        print("-" * 80)
    else:
        print("No experiments found.")
else:
    print(f"❌ Failed to list experiments: {list_response.status_code}")
    print(f"Error: {list_response.text}")

## 9. Summary and Next Steps

In [ ]:
print("🎉 AutoML Framework API Demo Complete!")
print("\n" + "="*60)
print("SUMMARY OF ACTIONS PERFORMED:")
print("="*60)
print("✅ 1. Successfully authenticated as admin user")
print("✅ 2. Checked API health and configuration")
print("✅ 3. Generated synthetic time series dataset")
print(f"✅ 4. Uploaded dataset to AutoML Framework" + (f" (ID: {dataset_id})" if dataset_id else " (FAILED)"))
print(f"✅ 5. Created time series forecasting experiment" + (f" (ID: {experiment_id})" if experiment_id else " (FAILED)"))
print("✅ 6. Monitored experiment progress")
print("✅ 7. Retrieved experiment results")
print("✅ 8. Listed all experiments")

print("\n" + "="*60)
print("NEXT STEPS:")
print("="*60)
print("🔄 1. Check experiment status periodically if still running")
print("📊 2. Analyze results and model performance when complete")
print("🚀 3. Deploy the best model for production forecasting")
print("🔧 4. Experiment with different configurations and parameters")
print("📈 5. Try different datasets and forecasting scenarios")

print("\n" + "="*60)
print("API ENDPOINTS USED:")
print("="*60)
print(f"• POST {API_BASE_URL}/api/v1/auth/login")
print(f"• GET  {API_BASE_URL}/health")
print(f"• GET  {API_BASE_URL}/api/v1/config")
print(f"• POST {API_BASE_URL}/api/v1/datasets/upload")
print(f"• POST {API_BASE_URL}/api/v1/experiments")
print(f"• GET  {API_BASE_URL}/api/v1/experiments/{{id}}")
print(f"• GET  {API_BASE_URL}/api/v1/experiments/{{id}}/results")
print(f"• GET  {API_BASE_URL}/api/v1/experiments")

print("\n🎯 Demo completed successfully! The AutoML Framework is ready for time series forecasting.")